In [1]:
import torch.nn as nn

class GRUClassification(nn.Module):
    def __init__(self, vocab_size, embed_dim=128,hidden_dim = 64):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.gru = nn.GRU(
            input_size= embed_dim,
            hidden_size = hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear( hidden_dim , 1)
    def forward(self, x):
        x = self.embedding(x)
        output, hidden =self.gru(x)
        # return output,hidden
        # hidden shape  (num_layer, batch, hidden_dim)   (1, batch , 64)        
        x = self.fc(hidden[-1])  # 2 dim 이고 마지막 압축된 은닉층 정보로 분류
        return x

In [2]:
import  torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter

In [3]:
# url = 'https://github.com/skn29teacher/skn29_lecture/blob/main/data_nlp/daum_movie_review.csv'
import pandas as pd
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
sample_df = df.iloc[-1000:]

In [4]:
sample_df['target'] = sample_df['rating'].apply(lambda x : 1 if x>0.5 else 0)

In [5]:

import re
def clean_text(text):
    return re.sub(r'[^가-힣\s]', '', text).strip()
df['clean'] = df['review'].apply(lambda x : clean_text( x ) )

In [6]:
#from konlpy.tag import Okt
# 공백을기준으로 분류
df['tokens'] = df['clean'].apply(lambda x : x.split() )

In [7]:
vocab_size = 10000
counter = Counter()
for tokens in df['tokens']:
    counter.update(tokens)
    
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1,
}    
for word, freq in counter.most_common(vocab_size):
    if len(word)>=2:
        vocab[word] = len(vocab)
print(f'단어수 : {len(vocab)}')

단어수 : 9712


In [8]:
vocab.items()

dict_items([('<PAD>', 0), ('<UNK>', 1), ('영화', 2), ('너무', 3), ('정말', 4), ('진짜', 5), ('그냥', 6), ('보고', 7), ('많이', 8), ('영화를', 9), ('연기', 10), ('영화가', 11), ('그리고', 12), ('있는', 13), ('영화는', 14), ('봤는데', 15), ('이런', 16), ('하는', 17), ('보는', 18), ('마지막', 19), ('재밌게', 20), ('좋은', 21), ('봤습니다', 22), ('스토리', 23), ('윤계상', 24), ('다시', 25), ('시간', 26), ('최고', 27), ('마동석', 28), ('감동', 29), ('이렇게', 30), ('없는', 31), ('재미있게', 32), ('눈물', 33), ('완전', 34), ('하지만', 35), ('역시', 36), ('많은', 37), ('눈물이', 38), ('함께', 39), ('내가', 40), ('봤어요', 41), ('보면', 42), ('최고의', 43), ('그래도', 44), ('연기가', 45), ('재미', 46), ('같은', 47), ('이게', 48), ('모두', 49), ('느낌', 50), ('하고', 51), ('내내', 52), ('근데', 53), ('없고', 54), ('대한', 55), ('배우들', 56), ('보세요', 57), ('조금', 58), ('어떻게', 59), ('마지막에', 60), ('다른', 61), ('없다', 62), ('솔직히', 63), ('영화입니다', 64), ('마블', 65), ('나름', 66), ('평점', 67), ('한국', 68), ('모르고', 69), ('영화의', 70), ('있고', 71), ('이건', 72), ('연기도', 73), ('한번', 74), ('영화다', 75), ('아주', 76), ('광주', 77), ('특히', 78), ('않고', 79)

In [9]:
from torch.nn.utils.rnn import pad_sequence
# 문장을 sequence_length 만큰 자른다.
# padd_sequence에 일괄 적용

sequence_length = 20
df['tokens_convert'] = df['tokens'].apply(
    lambda tokens : torch.LongTensor([ vocab.get(token,1) for token in tokens][:sequence_length] )
)
X = pad_sequence(df['tokens_convert'], batch_first=True, padding_value=0)
X.shape
y = df['rating'].apply(lambda x : x > 0.5).astype(int).values
y = torch.FloatTensor(y)
X.shape, y.shape


/tmp/ipykernel_14354/319055891.py:12: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  y = torch.FloatTensor(y)


(torch.Size([14725, 20]), torch.Size([14725]))

In [10]:
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)


In [11]:
x_train_dataset = TensorDataset(x_train,y_train)
x_train_dataloader = DataLoader(x_train_dataset,batch_size=32,shuffle=True)

x_test_dataset = TensorDataset(x_test,y_test)
x_test_dataloader = DataLoader(x_test_dataset,batch_size=32)


In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GRUClassification(len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [13]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_batch, y_batch in x_train_dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        pred = model(x_batch).squeeze(-1)        
        loss = criterion(pred, y_batch)        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()        
    print(
        f'Epoch:{epoch+1} '
        f'Loss:{total_loss/len(x_train_dataloader):.4f}'
    )

Epoch:1 Loss:0.2085
Epoch:2 Loss:0.1740
Epoch:3 Loss:0.1363
Epoch:4 Loss:0.0953
Epoch:5 Loss:0.0685


In [ ]:
model.eval()
with torch.no_grad():    
    total_loss = 0
    for x_batch, y_batch in x_test_dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        pred = model(x_batch).squeeze(-1)
        pred = torch.sigmoid(pred)
        print(pred, y_batch)
        break

In [ ]:
def predict(text):
    text = clean_text(text)
    tokens = text.split()
    encoded = ???
    padded = pad_sequence(encoded)
    tensor = torch.LongTensor([padded]).to(device)
    model.eval()
    with torch.no_grad():
        pred = model(tensor)
    score = pred.item()
    if score >= 0.5:
        print('긍정')
    else:
        print('부정')